<a href="https://colab.research.google.com/github/khunmyat17/Deep-cipher/blob/main/mes_vs_image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1 - GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

print("✅ CELL 1 READY - Google Drive Mounted")

Mounted at /content/drive
✅ CELL 1 READY - Google Drive Mounted


In [ ]:
# ============================================================
# CELL 2
# LOAD FINAL DEEP CIPHER MODEL
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


FINAL_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "DeepCipher_Checkpoints/"
    "deep_cipher_final.pth"
)

BACKUP_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "DeepCipher_Checkpoints/"
    "deep_cipher_v2_best.pth"
)


class DeepCipherV2(nn.Module):

    def __init__(self, secret_bits=128):

        super().__init__()

        self.secret_bits = secret_bits


        self.audio_encoder = nn.Sequential(

            nn.Conv1d(
                1, 32,
                kernel_size=15,
                stride=2,
                padding=7
            ),

            nn.BatchNorm1d(32),

            nn.LeakyReLU(0.2),

            nn.Conv1d(
                32, 64,
                kernel_size=15,
                stride=2,
                padding=7
            ),

            nn.BatchNorm1d(64),

            nn.LeakyReLU(0.2)
        )


        self.audio_decoder = nn.Sequential(

            nn.ConvTranspose1d(
                64, 32,
                kernel_size=16,
                stride=2,
                padding=7
            ),

            nn.BatchNorm1d(32),

            nn.ReLU(),

            nn.ConvTranspose1d(
                32, 1,
                kernel_size=16,
                stride=2,
                padding=7
            ),

            nn.Tanh()
        )


        self.secret_encoder = nn.Sequential(

            nn.Conv1d(
                1, 8,
                kernel_size=7,
                padding=3
            ),

            nn.LeakyReLU(0.2),

            nn.Conv1d(
                8, 8,
                kernel_size=7,
                padding=3
            ),

            nn.LeakyReLU(0.2)
        )


        self.fusion = nn.Sequential(

            nn.Conv1d(
                72, 64,
                kernel_size=5,
                padding=2
            ),

            nn.LeakyReLU(0.2),

            nn.Conv1d(
                64, 64,
                kernel_size=5,
                padding=2
            ),

            nn.Tanh()
        )


        self.embedding_strength = 0.04


        self.secret_decoder = nn.Sequential(

            nn.Conv1d(
                1, 32,
                kernel_size=15,
                stride=2,
                padding=7
            ),

            nn.LeakyReLU(0.2),

            nn.Conv1d(
                32, 64,
                kernel_size=15,
                stride=2,
                padding=7
            ),

            nn.LeakyReLU(0.2),

            nn.Conv1d(
                64, 64,
                kernel_size=15,
                stride=2,
                padding=7
            ),

            nn.LeakyReLU(0.2),

            nn.AdaptiveAvgPool1d(
                secret_bits
            ),

            nn.Conv1d(
                64, 1,
                kernel_size=1
            )
        )


    def forward(
        self,
        audio,
        secret_bits
    ):

        audio_latent = self.audio_encoder(
            audio
        )

        secret = secret_bits.unsqueeze(1)

        secret = F.interpolate(
            secret,
            size=audio_latent.shape[-1],
            mode="linear",
            align_corners=False
        )

        secret_features = self.secret_encoder(
            secret
        )

        combined = torch.cat(
            [
                audio_latent,
                secret_features
            ],
            dim=1
        )

        secret_delta = self.fusion(
            combined
        )

        stego_latent = (
            audio_latent
            +
            self.embedding_strength
            *
            secret_delta
        )

        stego_audio = self.audio_decoder(
            stego_latent
        )

        secret_logits = (
            self.secret_decoder(
                stego_audio
            )
            .squeeze(1)
        )

        return (
            stego_audio,
            secret_logits
        )


# ============================================================
# FIND CHECKPOINT
# ============================================================

if os.path.exists(FINAL_MODEL_PATH):

    MODEL_PATH = FINAL_MODEL_PATH

elif os.path.exists(BACKUP_MODEL_PATH):

    MODEL_PATH = BACKUP_MODEL_PATH
    print("⚠️ Using backup model")

else:

    raise FileNotFoundError(
        "❌ Deep Cipher checkpoint မတွေ့ပါ"
    )


print("Loading:", MODEL_PATH)


checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)


if isinstance(checkpoint, dict):

    secret_bits = int(
        checkpoint.get(
            "secret_bits",
            128
        )
    )

else:

    secret_bits = 128


deep_cipher_model = DeepCipherV2(
    secret_bits=secret_bits
).to(device)


if (
    isinstance(checkpoint, dict)
    and
    "model_state_dict" in checkpoint
):

    state_dict = checkpoint[
        "model_state_dict"
    ]

else:

    state_dict = checkpoint


deep_cipher_model.load_state_dict(
    state_dict,
    strict=True
)


if isinstance(checkpoint, dict):

    deep_cipher_model.embedding_strength = float(
        checkpoint.get(
            "embedding_strength",
            0.04
        )
    )


deep_cipher_model.eval()


# ============================================================
# TEST
# ============================================================

dummy_audio = torch.randn(
    1,
    1,
    48000,
    device=device
)

dummy_bits = torch.randint(
    0,
    2,
    (1, 128),
    device=device
).float()


with torch.no_grad():

    out_audio, out_bits = deep_cipher_model(
        dummy_audio,
        dummy_bits
    )


print("Audio Shape :", out_audio.shape)
print("Secret Shape:", out_bits.shape)

assert out_audio.shape == (1, 1, 48000)
assert out_bits.shape == (1, 128)

print(
    "\n✅ CELL 2 READY - Deep Cipher Model Loaded"
)

Device: cpu
Loading: /content/drive/MyDrive/DeepCipher_Checkpoints/deep_cipher_final.pth
Audio Shape : torch.Size([1, 1, 48000])
Secret Shape: torch.Size([1, 128])

✅ CELL 2 READY - Deep Cipher Model Loaded


In [ ]:
# ============================================================
# CELL 3
# DEEP CIPHER - MESSAGE + IMAGE IN ONE GUI
#
# AES-256-GCM
# PBKDF2-HMAC-SHA256
# CNN AUDIO STEGANOGRAPHY
# MESSAGE + IMAGE COMBINED PAYLOAD
# 16-BYTE CHUNKING
# ADAPTIVE EMBEDDING
# FAILED SEGMENT SKIP / RETRY
# FULL ORIGINAL AUDIO DURATION
# ============================================================

import os
import sys
import math
import time
import struct
import subprocess

import numpy as np
import torch
import soundfile as sf

from PIL import Image


# ============================================================
# INSTALL / IMPORT CRYPTOGRAPHY
# ============================================================

try:
    from cryptography.hazmat.primitives.ciphers.aead import AESGCM
    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
    from cryptography.hazmat.primitives import hashes
    from cryptography.exceptions import InvalidTag

except ImportError:

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "cryptography"
        ]
    )

    from cryptography.hazmat.primitives.ciphers.aead import AESGCM
    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
    from cryptography.hazmat.primitives import hashes
    from cryptography.exceptions import InvalidTag


# ============================================================
# LIBROSA
# ============================================================

try:
    import librosa

except ImportError:

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "librosa"
        ]
    )

    import librosa


# ============================================================
# GRADIO
# ============================================================

try:
    import gradio as gr

except ImportError:

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "gradio"
        ]
    )

    import gradio as gr


# ============================================================
# CHECK CELL 2
# ============================================================

if "deep_cipher_model" not in globals():

    raise RuntimeError(
        "❌ CELL 2 ကို အရင် Run ပါ။"
    )


if "device" not in globals():

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


deep_cipher_model = deep_cipher_model.to(device)

deep_cipher_model.eval()


# ============================================================
# AUDIO / MODEL SETTINGS
# ============================================================

SAMPLE_RATE = 16000

SEGMENT_SECONDS = 3

AUDIO_LENGTH = (
    SAMPLE_RATE
    *
    SEGMENT_SECONDS
)

SECRET_BITS = 128

CHUNK_BYTES = 16


# Message limit
MAX_MESSAGE_BYTES = 2000


# ============================================================
# ADAPTIVE EMBEDDING STRENGTH
# ============================================================

ADAPTIVE_STRENGTHS = [

    0.04,
    0.05,
    0.06,
    0.075,
    0.10,
    0.125,
    0.15,
    0.175,
    0.20
]


# ============================================================
# AES SETTINGS
# ============================================================

AES_KEY_BYTES = 32

SALT_BYTES = 16

NONCE_BYTES = 12

AES_TAG_BYTES = 16

PBKDF2_ITERATIONS = 200000


# Outer encrypted packet
PACKET_MAGIC = b"DCMX"

PACKET_VERSION = 1


# 4 magic
# 1 version
# 16 salt
# 12 nonce
# 4 cipher length

PACKET_HEADER_SIZE = (
    4
    +
    1
    +
    SALT_BYTES
    +
    NONCE_BYTES
    +
    4
)


# ============================================================
# INNER MESSAGE + IMAGE PAYLOAD
# ============================================================

PAYLOAD_MAGIC = b"PAY1"


# Payload:
#
# MAGIC       4
# FLAGS       1
# MSG LENGTH  4
# WIDTH       1
# HEIGHT      1
# IMAGE LEN   4
#
# TOTAL = 15 bytes

PAYLOAD_HEADER_SIZE = (
    4
    +
    1
    +
    4
    +
    1
    +
    1
    +
    4
)


FLAG_MESSAGE = 1

FLAG_IMAGE = 2


# ============================================================
# OUTPUT
# ============================================================

OUTPUT_FOLDER = (
    "/content/drive/MyDrive/"
    "DeepCipher_Checkpoints/"
    "Combined_Message_Image_Output"
)


os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)


# ============================================================
# IMAGE SIZE
# ============================================================

def parse_image_size(
    option
):

    option = str(option)

    if option.startswith("24"):
        return 24

    if option.startswith("16"):
        return 16

    return 8


# ============================================================
# IMAGE PREPARATION
# ============================================================

def prepare_image(
    image_path,
    side
):

    if not image_path:

        return (
            b"",
            None,
            0,
            0
        )


    image = Image.open(
        image_path
    )


    # Grayscale
    image = image.convert(
        "L"
    )


    image = image.resize(
        (
            side,
            side
        ),
        Image.Resampling.LANCZOS
    )


    image_array = np.asarray(
        image,
        dtype=np.uint8
    )


    raw_bytes = (
        image_array.tobytes()
    )


    return (
        raw_bytes,
        image,
        side,
        side
    )


# ============================================================
# BUILD MESSAGE + IMAGE PAYLOAD
# ============================================================

def build_payload(
    message,
    image_path,
    image_size_option
):

    message = (
        message
        if message is not None
        else ""
    )


    message_bytes = message.encode(
        "utf-8"
    )


    if len(message_bytes) > MAX_MESSAGE_BYTES:

        raise ValueError(
            f"Message is too long.\n"
            f"Maximum = {MAX_MESSAGE_BYTES} bytes\n"
            f"Current = {len(message_bytes)} bytes"
        )


    side = parse_image_size(
        image_size_option
    )


    image_bytes, tiny_image, width, height = (
        prepare_image(
            image_path,
            side
        )
    )


    # At least one input
    if (
        len(message_bytes) == 0
        and
        len(image_bytes) == 0
    ):

        raise ValueError(
            "Message သို့မဟုတ် Image "
            "အနည်းဆုံးတစ်ခု ထည့်ပါ။"
        )


    flags = 0


    if len(message_bytes) > 0:

        flags |= FLAG_MESSAGE


    if len(image_bytes) > 0:

        flags |= FLAG_IMAGE


    payload_header = (

        PAYLOAD_MAGIC

        +

        bytes(
            [flags]
        )

        +

        struct.pack(
            ">I",
            len(message_bytes)
        )

        +

        bytes(
            [width]
        )

        +

        bytes(
            [height]
        )

        +

        struct.pack(
            ">I",
            len(image_bytes)
        )
    )


    payload = (

        payload_header

        +

        message_bytes

        +

        image_bytes
    )


    return (
        payload,
        tiny_image,
        len(message_bytes),
        len(image_bytes),
        width,
        height
    )


# ============================================================
# PARSE DECRYPTED PAYLOAD
# ============================================================

def parse_payload(
    payload
):

    if len(payload) < PAYLOAD_HEADER_SIZE:

        raise ValueError(
            "Payload is incomplete."
        )


    offset = 0


    magic = payload[
        offset:
        offset + 4
    ]

    offset += 4


    if magic != PAYLOAD_MAGIC:

        raise ValueError(
            "Invalid combined payload."
        )


    flags = payload[
        offset
    ]

    offset += 1


    message_len = struct.unpack(
        ">I",
        payload[
            offset:
            offset + 4
        ]
    )[0]

    offset += 4


    width = payload[
        offset
    ]

    offset += 1


    height = payload[
        offset
    ]

    offset += 1


    image_len = struct.unpack(
        ">I",
        payload[
            offset:
            offset + 4
        ]
    )[0]

    offset += 4


    required_size = (

        PAYLOAD_HEADER_SIZE

        +

        message_len

        +

        image_len
    )


    if len(payload) < required_size:

        raise ValueError(
            "Recovered payload is incomplete."
        )


    message_bytes = payload[
        offset:
        offset + message_len
    ]

    offset += message_len


    image_bytes = payload[
        offset:
        offset + image_len
    ]


    # Message
    if flags & FLAG_MESSAGE:

        message = message_bytes.decode(
            "utf-8"
        )

    else:

        message = ""


    # Image validation
    if flags & FLAG_IMAGE:

        if (
            width not in [8, 16, 24]
            or
            height not in [8, 16, 24]
        ):

            raise ValueError(
                "Invalid recovered image size."
            )


        if image_len != (
            width
            *
            height
        ):

            raise ValueError(
                "Recovered image data size mismatch."
            )


    return (
        message,
        image_bytes,
        width,
        height,
        flags
    )


# ============================================================
# PBKDF2-HMAC-SHA256
# ============================================================

def derive_aes_key(
    password,
    salt
):

    if not password:

        raise ValueError(
            "AES Secret Key cannot be empty."
        )


    kdf = PBKDF2HMAC(

        algorithm=hashes.SHA256(),

        length=AES_KEY_BYTES,

        salt=salt,

        iterations=PBKDF2_ITERATIONS
    )


    return kdf.derive(
        password.encode(
            "utf-8"
        )
    )


# ============================================================
# AES-256-GCM ENCRYPT
# ============================================================

def aes_encrypt(
    payload,
    password
):

    salt = os.urandom(
        SALT_BYTES
    )


    nonce = os.urandom(
        NONCE_BYTES
    )


    key = derive_aes_key(
        password,
        salt
    )


    cipher_len = (
        len(payload)
        +
        AES_TAG_BYTES
    )


    header = (

        PACKET_MAGIC

        +

        bytes(
            [PACKET_VERSION]
        )

        +

        salt

        +

        nonce

        +

        struct.pack(
            ">I",
            cipher_len
        )
    )


    aes = AESGCM(
        key
    )


    ciphertext = aes.encrypt(

        nonce,

        payload,

        header
    )


    return (
        header
        +
        ciphertext
    )


# ============================================================
# AES-256-GCM DECRYPT
# ============================================================

def aes_decrypt(
    packet,
    password
):

    try:

        if len(packet) < PACKET_HEADER_SIZE:

            return (
                None,
                "❌ Incomplete encrypted packet."
            )


        offset = 0


        magic = packet[
            offset:
            offset + 4
        ]

        offset += 4


        if magic != PACKET_MAGIC:

            return (
                None,
                "❌ Invalid Deep Cipher packet."
            )


        version = packet[
            offset
        ]

        offset += 1


        if version != PACKET_VERSION:

            return (
                None,
                "❌ Unsupported packet version."
            )


        salt = packet[
            offset:
            offset + SALT_BYTES
        ]

        offset += SALT_BYTES


        nonce = packet[
            offset:
            offset + NONCE_BYTES
        ]

        offset += NONCE_BYTES


        cipher_len = struct.unpack(
            ">I",
            packet[
                offset:
                offset + 4
            ]
        )[0]


        total_size = (
            PACKET_HEADER_SIZE
            +
            cipher_len
        )


        if len(packet) < total_size:

            return (
                None,
                "❌ Encrypted packet is incomplete."
            )


        header = packet[
            :PACKET_HEADER_SIZE
        ]


        ciphertext = packet[
            PACKET_HEADER_SIZE:
            total_size
        ]


        key = derive_aes_key(
            password,
            salt
        )


        aes = AESGCM(
            key
        )


        plaintext = aes.decrypt(

            nonce,

            ciphertext,

            header
        )


        return (
            plaintext,
            "✅ AES-256-GCM authentication verified."
        )


    except InvalidTag:

        return (
            None,
            (
                "❌ ACCESS DENIED\n"
                "Wrong Secret Key or hidden data was corrupted."
            )
        )


    except Exception as e:

        return (
            None,
            f"❌ AES Error: {e}"
        )


# ============================================================
# PACKET -> CHUNKS
# ============================================================

def packet_to_chunks(
    packet
):

    chunks = []


    for start in range(
        0,
        len(packet),
        CHUNK_BYTES
    ):

        chunk = packet[
            start:
            start + CHUNK_BYTES
        ]


        if len(chunk) < CHUNK_BYTES:

            chunk = chunk.ljust(
                CHUNK_BYTES,
                b"\x00"
            )


        chunks.append(
            chunk
        )


    return chunks


# ============================================================
# 16 BYTES -> 128 BITS
# ============================================================

def chunk_to_bits(
    chunk
):

    chunk = chunk.ljust(
        CHUNK_BYTES,
        b"\x00"
    )[:CHUNK_BYTES]


    bits = []


    for value in chunk:

        for position in range(
            7,
            -1,
            -1
        ):

            bits.append(
                (
                    value
                    >>
                    position
                )
                &
                1
            )


    return torch.tensor(
        bits,
        dtype=torch.float32
    ).unsqueeze(0)


# ============================================================
# 128 BITS -> 16 BYTES
# ============================================================

def bits_to_chunk(
    bits
):

    bits = (
        bits
        .detach()
        .cpu()
        .flatten()
        .int()
        .tolist()
    )


    output = bytearray()


    for start in range(
        0,
        SECRET_BITS,
        8
    ):

        value = 0


        for bit in bits[
            start:
            start + 8
        ]:

            value = (
                value << 1
            ) | int(bit)


        output.append(
            value
        )


    return bytes(
        output
    )


# ============================================================
# COVER AUDIO
# ============================================================

def load_cover_audio(
    filepath
):

    if not filepath:

        raise ValueError(
            "Please upload Cover Audio."
        )


    audio_np, _ = librosa.load(

        filepath,

        sr=SAMPLE_RATE,

        mono=True
    )


    audio_np = np.asarray(
        audio_np,
        dtype=np.float32
    )


    return torch.from_numpy(
        audio_np
    ).unsqueeze(0)


# ============================================================
# NORMALIZE 3-SECOND COVER SEGMENT
# ============================================================

def normalize_segment(
    segment
):

    segment = segment.float()


    peak = segment.abs().max()


    if peak > 1e-8:

        segment = (

            segment

            /

            peak

            *

            0.95
        )


    return segment


# ============================================================
# SNR
# ============================================================

def calculate_snr(
    cover,
    stego
):

    cover = (
        cover
        .detach()
        .cpu()
        .float()
    )


    stego = (
        stego
        .detach()
        .cpu()
        .float()
    )


    signal_power = torch.mean(
        cover ** 2
    )


    noise_power = torch.mean(
        (
            cover
            -
            stego
        )
        **
        2
    )


    return float(
        (
            10
            *
            torch.log10(
                (
                    signal_power
                    +
                    1e-12
                )
                /
                (
                    noise_power
                    +
                    1e-12
                )
            )
        ).item()
    )


# ============================================================
# ENCODE ONE CHUNK
# ============================================================

def encode_one_chunk(
    cover_segment,
    chunk
):

    target_bits = (
        chunk_to_bits(
            chunk
        )
        .to(device)
    )


    cover_segment = (
        normalize_segment(
            cover_segment
        )
        .to(device)
    )


    old_strength = float(
        deep_cipher_model.embedding_strength
    )


    successes = []

    best_failure = None


    try:

        for strength in ADAPTIVE_STRENGTHS:

            deep_cipher_model.embedding_strength = (
                strength
            )


            with torch.no_grad():

                stego_audio, logits = (
                    deep_cipher_model(
                        cover_segment,
                        target_bits
                    )
                )


            recovered = (
                torch.sigmoid(
                    logits
                )
                >=
                0.5
            ).float()


            errors = int(
                (
                    recovered
                    !=
                    target_bits
                )
                .sum()
                .item()
            )


            ber = (
                errors
                /
                SECRET_BITS
            )


            accuracy = (
                100.0
                *
                (
                    1.0
                    -
                    ber
                )
            )


            snr = calculate_snr(
                cover_segment,
                stego_audio
            )


            result = {

                "success":
                    errors == 0,

                "stego":
                    stego_audio
                    .detach()
                    .cpu(),

                "strength":
                    strength,

                "accuracy":
                    accuracy,

                "ber":
                    ber,

                "errors":
                    errors,

                "snr":
                    snr
            }


            if errors == 0:

                successes.append(
                    result
                )


            if (
                best_failure is None
                or
                result["ber"]
                <
                best_failure["ber"]
            ):

                best_failure = result


            elif (
                result["ber"]
                ==
                best_failure["ber"]

                and

                result["snr"]
                >
                best_failure["snr"]
            ):

                best_failure = result


        if successes:

            return max(
                successes,
                key=lambda x:
                    x["snr"]
            )


        return best_failure


    finally:

        deep_cipher_model.embedding_strength = (
            old_strength
        )


# ============================================================
# EMBED COMPLETE PACKET
# ============================================================

def adaptive_embed_packet(
    cover_audio,
    packet
):

    chunks = packet_to_chunks(
        packet
    )


    required_chunks = len(
        chunks
    )


    total_samples = int(
        cover_audio.shape[-1]
    )


    original_duration = (
        total_samples
        /
        SAMPLE_RATE
    )


    available_segments = (
        total_samples
        //
        AUDIO_LENGTH
    )


    if available_segments < required_chunks:

        return (
            None,
            (
                "❌ COVER AUDIO TOO SHORT\n\n"
                f"Required Chunks: {required_chunks}\n"
                f"Available Segments: {available_segments}\n"
                f"Theoretical Minimum: "
                f"{required_chunks * SEGMENT_SECONDS} sec\n\n"
                "Please use a longer Cover Audio."
            )
        )


    successful_segments = []


    chunk_index = 0

    cover_index = 0


    best_accuracy = 0

    best_ber = 1


    while (
        chunk_index < required_chunks
        and
        cover_index < available_segments
    ):

        start = (
            cover_index
            *
            AUDIO_LENGTH
        )


        end = (
            start
            +
            AUDIO_LENGTH
        )


        segment = (
            cover_audio[
                :,
                start:end
            ]
            .unsqueeze(0)
        )


        print(
            f"\nChunk "
            f"{chunk_index + 1}/{required_chunks}"
            f" → Cover Segment "
            f"{cover_index + 1}/{available_segments}"
        )


        result = encode_one_chunk(
            segment,
            chunks[
                chunk_index
            ]
        )


        print(
            f"Accuracy : {result['accuracy']:.2f}%"
        )

        print(
            f"BER      : {result['ber']:.6f}"
        )

        print(
            f"Errors   : {result['errors']}"
        )

        print(
            f"Strength : {result['strength']}"
        )

        print(
            f"SNR      : {result['snr']:.2f} dB"
        )


        if result["success"]:

            print(
                "✅ ACCEPT"
            )


            successful_segments.append(
                result["stego"]
            )


            chunk_index += 1


            best_accuracy = 0

            best_ber = 1


        else:

            print(
                "❌ SKIP → Retry same chunk"
            )


            best_accuracy = max(
                best_accuracy,
                result["accuracy"]
            )


            best_ber = min(
                best_ber,
                result["ber"]
            )


        cover_index += 1


    if chunk_index < required_chunks:

        return (
            None,
            (
                "❌ EMBEDDING FAILED AFTER RETRIES\n\n"
                f"Successful Chunks: "
                f"{chunk_index}/{required_chunks}\n"
                f"Failed Chunk: "
                f"{chunk_index + 1}/{required_chunks}\n"
                f"Cover Segments Tested: "
                f"{cover_index}/{available_segments}\n"
                f"Best Accuracy: "
                f"{best_accuracy:.2f}%\n"
                f"Best BER: "
                f"{best_ber:.6f}\n\n"
                "Try a longer or different Cover Audio."
            )
        )


    # ========================================================
    # SUCCESSFUL HIDDEN PART
    # ========================================================

    hidden_part = torch.cat(
        successful_segments,
        dim=2
    ).squeeze(0).cpu()


    hidden_samples = int(
        hidden_part.shape[-1]
    )


    # ========================================================
    # KEEP ORIGINAL AUDIO DURATION
    # ========================================================

    final_stego = (
        cover_audio
        .detach()
        .cpu()
        .clone()
    )


    final_stego[
        :,
        :hidden_samples
    ] = hidden_part


    final_stego = final_stego[
        :,
        :total_samples
    ]


    final_duration = (
        final_stego.shape[-1]
        /
        SAMPLE_RATE
    )


    skipped = (
        cover_index
        -
        required_chunks
    )


    return (
        final_stego,
        {
            "required_chunks":
                required_chunks,

            "available_segments":
                available_segments,

            "tested_segments":
                cover_index,

            "skipped_segments":
                skipped,

            "original_duration":
                original_duration,

            "final_duration":
                final_duration,

            "hidden_duration":
                required_chunks
                *
                SEGMENT_SECONDS
        }
    )


# ============================================================
# CAPACITY CALCULATOR
# ============================================================

def calculate_capacity(
    message,
    image_file,
    image_size_option
):

    try:

        message = (
            message
            if message is not None
            else ""
        )


        message_bytes = len(
            message.encode(
                "utf-8"
            )
        )


        if image_file:

            side = parse_image_size(
                image_size_option
            )

            image_bytes = (
                side
                *
                side
            )

        else:

            side = 0

            image_bytes = 0


        payload_bytes = (
            PAYLOAD_HEADER_SIZE
            +
            message_bytes
            +
            image_bytes
        )


        packet_bytes = (
            PACKET_HEADER_SIZE
            +
            payload_bytes
            +
            AES_TAG_BYTES
        )


        chunks = math.ceil(
            packet_bytes
            /
            CHUNK_BYTES
        )


        seconds = (
            chunks
            *
            SEGMENT_SECONDS
        )


        minutes = (
            seconds
            //
            60
        )


        remaining = (
            seconds
            %
            60
        )


        image_text = (
            f"{side}x{side}"
            if image_file
            else "None"
        )


        return (
            "MESSAGE + IMAGE CAPACITY\n"
            "===================================\n"
            f"Message Bytes        : {message_bytes}\n"
            f"Image Size           : {image_text}\n"
            f"Image Pixel Bytes    : {image_bytes}\n"
            f"Combined Payload     : {payload_bytes} bytes\n"
            f"AES Packet Bytes     : {packet_bytes}\n"
            f"16-byte Chunks       : {chunks}\n"
            f"Minimum Audio        : {seconds} sec\n"
            f"Approximately        : "
            f"{minutes} min {remaining} sec\n\n"
            "⚠️ Adaptive Retry အတွက် theoretical "
            "minimum ထက် ပိုရှည်တဲ့ audio သုံးပါ။"
        )


    except Exception as e:

        return (
            f"❌ Capacity Error:\n{e}"
        )


# ============================================================
# SENDER
# ============================================================

def sender_process(
    cover_file,
    message,
    image_file,
    image_size_option,
    secret_key
):

    try:

        if not cover_file:

            return (
                None,
                None,
                "❌ Please upload Cover Audio."
            )


        if not secret_key:

            return (
                None,
                None,
                "❌ Please enter AES Secret Key."
            )


        # ====================================================
        # COMBINE MESSAGE + IMAGE
        # ====================================================

        (
            payload,
            tiny_image,
            message_bytes,
            image_bytes,
            width,
            height
        ) = build_payload(
            message,
            image_file,
            image_size_option
        )


        # ====================================================
        # PREVIEW EXACT IMAGE BEING HIDDEN
        # ====================================================

        preview_path = None


        timestamp = time.strftime(
            "%Y%m%d_%H%M%S"
        )


        if tiny_image is not None:

            preview_path = os.path.join(
                OUTPUT_FOLDER,
                f"sender_image_preview_{timestamp}.png"
            )


            # Visual enlargement only
            tiny_image.resize(
                (
                    width * 20,
                    height * 20
                ),
                Image.Resampling.NEAREST
            ).save(
                preview_path
            )


        # ====================================================
        # AES ENCRYPT COMBINED PAYLOAD
        # ====================================================

        packet = aes_encrypt(
            payload,
            secret_key
        )


        # ====================================================
        # COVER AUDIO
        # ====================================================

        cover_audio = load_cover_audio(
            cover_file
        )


        # ====================================================
        # CNN EMBEDDING
        # ====================================================

        final_stego, result = (
            adaptive_embed_packet(
                cover_audio,
                packet
            )
        )


        if final_stego is None:

            return (
                None,
                preview_path,
                result
            )


        # ====================================================
        # SAVE FULL-LENGTH FLOAT32 WAV
        # ====================================================

        output_path = os.path.join(
            OUTPUT_FOLDER,
            f"deepcipher_message_image_{timestamp}.wav"
        )


        sf.write(

            output_path,

            final_stego
            .squeeze(0)
            .numpy()
            .astype(
                np.float32
            ),

            SAMPLE_RATE,

            subtype="FLOAT"
        )


        status = (

            "✅ MESSAGE + IMAGE EMBEDDING SUCCESS\n\n"

            f"Message Bytes: "
            f"{message_bytes}\n"

            f"Image Bytes: "
            f"{image_bytes}\n"

            f"Image Size: "
            f"{width}x{height}"
            if image_bytes > 0
            else
            "Image: None"
        )


        status += (

            "\n\n"

            f"AES Packet Bytes: "
            f"{len(packet)}\n"

            f"AES Chunks: "
            f"{result['required_chunks']}\n"

            f"Cover Segments Tested: "
            f"{result['tested_segments']}/"
            f"{result['available_segments']}\n"

            f"Skipped Segments: "
            f"{result['skipped_segments']}\n\n"

            f"Hidden Data Portion: "
            f"{result['hidden_duration']:.2f} sec\n"

            f"Original Audio Duration: "
            f"{result['original_duration']:.2f} sec\n"

            f"Final Output Duration: "
            f"{result['final_duration']:.2f} sec\n\n"

            "✅ Original Audio Duration Preserved\n"

            "✅ AES-256-GCM\n"

            "✅ CNN Audio Steganography"
        )


        return (
            output_path,
            preview_path,
            status
        )


    except Exception as e:

        return (
            None,
            None,
            f"❌ SENDER ERROR:\n{e}"
        )


# ============================================================
# LOAD STEGO EXACTLY
# ============================================================

def load_stego_exact(
    filepath
):

    audio, sr = sf.read(

        filepath,

        dtype="float32",

        always_2d=True
    )


    audio = audio.T


    if sr != SAMPLE_RATE:

        raise ValueError(
            "Stego WAV must be 16000 Hz."
        )


    if audio.shape[0] != 1:

        raise ValueError(
            "Stego WAV must be Mono."
        )


    return torch.tensor(
        audio,
        dtype=torch.float32
    )


# ============================================================
# DIRECT CNN SECRET DECODER
# ============================================================

def decode_one_segment(
    segment
):

    segment = segment.to(
        device
    )


    with torch.no_grad():

        logits = (
            deep_cipher_model
            .secret_decoder(
                segment
            )
            .squeeze(1)
        )


        probabilities = torch.sigmoid(
            logits
        )


        bits = (
            probabilities
            >=
            0.5
        ).float()


        confidence = (
            torch.where(
                bits > 0.5,
                probabilities,
                1.0 - probabilities
            )
            .mean()
            .item()
            *
            100
        )


    return (
        bits,
        confidence
    )


# ============================================================
# RECEIVER
# ============================================================

def receiver_process(
    stego_file,
    secret_key
):

    try:

        if not stego_file:

            return (
                "",
                None,
                None,
                "❌ Please upload Stego WAV."
            )


        if not secret_key:

            return (
                "",
                None,
                None,
                "❌ Please enter AES Secret Key."
            )


        stego_audio = load_stego_exact(
            stego_file
        )


        available_segments = (
            stego_audio.shape[-1]
            //
            AUDIO_LENGTH
        )


        total_audio_duration = (
            stego_audio.shape[-1]
            /
            SAMPLE_RATE
        )


        recovered_bytes = bytearray()

        confidence_values = []


        required_chunks = None

        total_packet_size = None


        # ====================================================
        # DECODE CONTIGUOUS HIDDEN SEGMENTS
        # ====================================================

        for segment_index in range(
            available_segments
        ):

            start = (
                segment_index
                *
                AUDIO_LENGTH
            )


            end = (
                start
                +
                AUDIO_LENGTH
            )


            segment = (
                stego_audio[
                    :,
                    start:end
                ]
                .unsqueeze(0)
            )


            bits, confidence = (
                decode_one_segment(
                    segment
                )
            )


            recovered_bytes.extend(
                bits_to_chunk(
                    bits[0]
                )
            )


            confidence_values.append(
                confidence
            )


            print(
                f"Decoded Segment "
                f"{segment_index + 1}"
                f" | Confidence "
                f"{confidence:.2f}%"
            )


            # =================================================
            # OUTER AES HEADER AVAILABLE
            # =================================================

            if (
                required_chunks is None
                and
                len(recovered_bytes)
                >=
                PACKET_HEADER_SIZE
            ):

                header = bytes(
                    recovered_bytes[
                        :PACKET_HEADER_SIZE
                    ]
                )


                if header[:4] != PACKET_MAGIC:

                    return (
                        "",
                        None,
                        None,
                        (
                            "❌ INVALID / CORRUPTED STEGO AUDIO\n"
                            "Deep Cipher packet header "
                            "could not be recovered."
                        )
                    )


                if header[4] != PACKET_VERSION:

                    return (
                        "",
                        None,
                        None,
                        "❌ Invalid Packet Version."
                    )


                cipher_position = (
                    4
                    +
                    1
                    +
                    SALT_BYTES
                    +
                    NONCE_BYTES
                )


                cipher_len = struct.unpack(
                    ">I",
                    header[
                        cipher_position:
                        cipher_position + 4
                    ]
                )[0]


                # Safety
                if cipher_len > 10000:

                    return (
                        "",
                        None,
                        None,
                        "❌ Corrupted packet length."
                    )


                total_packet_size = (
                    PACKET_HEADER_SIZE
                    +
                    cipher_len
                )


                required_chunks = math.ceil(
                    total_packet_size
                    /
                    CHUNK_BYTES
                )


                print(
                    "Required Chunks:",
                    required_chunks
                )


            # Stop after encrypted packet
            if (
                required_chunks is not None
                and
                (
                    segment_index + 1
                )
                >=
                required_chunks
            ):

                break


        if required_chunks is None:

            return (
                "",
                None,
                None,
                "❌ AES packet header not recovered."
            )


        if len(confidence_values) < required_chunks:

            return (
                "",
                None,
                None,
                (
                    "❌ Incomplete hidden data.\n"
                    f"Required: {required_chunks}\n"
                    f"Decoded: {len(confidence_values)}"
                )
            )


        packet = bytes(
            recovered_bytes[
                :total_packet_size
            ]
        )


        # ====================================================
        # AES DECRYPT
        # ====================================================

        payload, aes_status = aes_decrypt(
            packet,
            secret_key
        )


        average_confidence = (
            sum(confidence_values)
            /
            len(confidence_values)
        )


        if payload is None:

            return (
                "",
                None,
                None,
                (
                    f"{aes_status}\n\n"
                    f"Average CNN Confidence: "
                    f"{average_confidence:.2f}%"
                )
            )


        # ====================================================
        # SEPARATE MESSAGE + IMAGE
        # ====================================================

        (
            message,
            image_bytes,
            width,
            height,
            flags
        ) = parse_payload(
            payload
        )


        recovered_preview_path = None

        recovered_image_path = None


        timestamp = time.strftime(
            "%Y%m%d_%H%M%S"
        )


        # ====================================================
        # REBUILD IMAGE
        # ====================================================

        if (
            flags & FLAG_IMAGE
            and
            len(image_bytes) > 0
        ):

            image_array = np.frombuffer(
                image_bytes,
                dtype=np.uint8
            ).reshape(
                height,
                width
            )


            recovered_image = Image.fromarray(
                image_array
            )


            recovered_image_path = os.path.join(
                OUTPUT_FOLDER,
                f"recovered_exact_{timestamp}.png"
            )


            recovered_image.save(
                recovered_image_path
            )


            # Large preview
            recovered_preview_path = os.path.join(
                OUTPUT_FOLDER,
                f"recovered_preview_{timestamp}.png"
            )


            recovered_image.resize(
                (
                    width * 20,
                    height * 20
                ),
                Image.Resampling.NEAREST
            ).save(
                recovered_preview_path
            )


        status = (

            "✅ MESSAGE + IMAGE RECOVERED SUCCESSFULLY\n\n"

            f"{aes_status}\n"

            f"Decoded Hidden Segments: "
            f"{required_chunks}\n"

            f"Full Stego Duration: "
            f"{total_audio_duration:.2f} sec\n"

            f"Average CNN Confidence: "
            f"{average_confidence:.2f}%\n\n"
        )


        if flags & FLAG_MESSAGE:

            status += (
                "✅ Secret Message Recovered\n"
            )


        if flags & FLAG_IMAGE:

            status += (
                f"✅ Secret Image Recovered "
                f"({width}x{height})\n"
            )


        return (
            message,
            recovered_preview_path,
            recovered_image_path,
            status
        )


    except Exception as e:

        return (
            "",
            None,
            None,
            f"❌ RECEIVER ERROR:\n{e}"
        )


# ============================================================
# AES COMBINED PAYLOAD SELF TEST
# ============================================================

test_payload = (

    PAYLOAD_MAGIC
    +
    bytes([FLAG_MESSAGE])
    +
    struct.pack(">I", 5)
    +
    bytes([0])
    +
    bytes([0])
    +
    struct.pack(">I", 0)
    +
    b"HELLO"
)


test_packet = aes_encrypt(
    test_payload,
    "Key@2026"
)


test_result, _ = aes_decrypt(
    test_packet,
    "Key@2026"
)


assert (
    test_result
    ==
    test_payload
)


print(
    "✅ MESSAGE + IMAGE AES SELF-TEST PASSED"
)


# ============================================================
# GUI
# ============================================================

APP_USERNAME = "deepcipher"

APP_PASSWORD = "DeepCipher2026"


with gr.Blocks(
    title="Deep Cipher"
) as demo:


    gr.Markdown(
        """
# 🔐 Deep Cipher
### Message + Image Audio Steganography

**One GUI → Message + Image can be hidden together**

AES-256-GCM + CNN Autoencoder
"""
    )


    # ========================================================
    # SENDER
    # ========================================================

    with gr.Tab(
        "📤 Sender"
    ):


        cover_input = gr.File(

            label="Cover Audio",

            file_types=[
                ".wav",
                ".mp3",
                ".flac",
                ".ogg"
            ],

            type="filepath"
        )


        message_input = gr.Textbox(

            label="Secret Message",

            lines=6,

            placeholder=(
                "Enter Secret Message..."
            )
        )


        image_input = gr.Image(

            label="Secret Image",

            type="filepath"
        )


        image_size = gr.Dropdown(

            choices=[

                "8x8 - Small / Fast",

                "16x16 - Recommended",

                "24x24 - Higher Detail / Longer Audio"

            ],

            value="16x16 - Recommended",

            label="Hidden Image Resolution"
        )


        sender_key = gr.Textbox(

            label="AES Secret Key",

            type="password",

            placeholder=(
                "Enter Secret Key..."
            )
        )


        capacity_button = gr.Button(
            "📏 Calculate Required Audio"
        )


        capacity_output = gr.Textbox(

            label="Capacity Information",

            lines=10
        )


        send_button = gr.Button(
            "🔐 Encrypt & Hide Message + Image"
        )


        stego_output = gr.File(
            label="Full-Length Stego WAV"
        )


        hidden_image_preview = gr.Image(
            label="Image Actually Hidden"
        )


        sender_status = gr.Textbox(

            label="Sender Status",

            lines=15
        )


        capacity_button.click(

            fn=calculate_capacity,

            inputs=[
                message_input,
                image_input,
                image_size
            ],

            outputs=[
                capacity_output
            ]
        )


        send_button.click(

            fn=sender_process,

            inputs=[
                cover_input,
                message_input,
                image_input,
                image_size,
                sender_key
            ],

            outputs=[
                stego_output,
                hidden_image_preview,
                sender_status
            ]
        )


    # ========================================================
    # RECEIVER
    # ========================================================

    with gr.Tab(
        "📥 Receiver"
    ):


        gr.Markdown(
            """
Upload the **exact generated Stego WAV** and enter the same AES key.

The system will recover **both Message and Image**.
"""
        )


        receiver_stego = gr.File(

            label="Stego WAV",

            file_types=[
                ".wav"
            ],

            type="filepath"
        )


        receiver_key = gr.Textbox(

            label="AES Secret Key",

            type="password",

            placeholder=(
                "Enter Same Secret Key..."
            )
        )


        receive_button = gr.Button(
            "🔓 Extract Message + Image"
        )


        recovered_message = gr.Textbox(

            label="Recovered Secret Message",

            lines=6
        )


        recovered_image = gr.Image(

            label="Recovered Secret Image"
        )


        recovered_png = gr.File(

            label="Recovered PNG File"
        )


        receiver_status = gr.Textbox(

            label="Receiver Status",

            lines=12
        )


        receive_button.click(

            fn=receiver_process,

            inputs=[
                receiver_stego,
                receiver_key
            ],

            outputs=[
                recovered_message,
                recovered_image,
                recovered_png,
                receiver_status
            ]
        )


# ============================================================
# READY
# ============================================================

print("\n========================================")
print("✅ CELL 3 READY")
print("========================================")

print("✅ Message + Image in ONE payload")
print("✅ AES-256-GCM")
print("✅ PBKDF2-HMAC-SHA256")
print("✅ CNN Audio Steganography")
print("✅ Adaptive Segment Retry")
print("✅ Full Audio Duration Preserved")
print("✅ Message Recovery")
print("✅ Image Recovery")


# ============================================================
# LAUNCH
# ============================================================

demo.launch(

    share=True,

    auth=(
        APP_USERNAME,
        APP_PASSWORD
    )
)

✅ MESSAGE + IMAGE AES SELF-TEST PASSED

✅ CELL 3 READY
✅ Message + Image in ONE payload
✅ AES-256-GCM
✅ PBKDF2-HMAC-SHA256
✅ CNN Audio Steganography
✅ Adaptive Segment Retry
✅ Full Audio Duration Preserved
✅ Message Recovery
✅ Image Recovery
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f46f676de53569eeae.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
